# Box-constrained CUTEst

This notebook benchmarks projected NTRQN on regular CUTEst problems whose only constraints are variable bounds. Results are stored separately under `data/temp/boxed`.

In [ ]:
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.experiment.for_cutest_run import load_results, run
from qnlab.experiment.for_cutest_vis import draw_pp
from qnlab.util.method import get_box_methods

In [ ]:
os.chdir(Path(os.path.abspath("cutest_boxed.ipynb")).parent.parent.resolve())
print(os.getcwd())

In [ ]:
precision = 64
noise = np.float64(0.0)
gtol = np.float64(1e-5)
time_limit = 600

# CHEBYQADNE (which has no objective) is excluded by problemsToRun.
problems = [
    p
    for p in problemsToRun(None, constraints="bound")
    if p not in ["WALL50", "WALL100"]
]
assert "CHEBYQADNE" not in problems, "CHEBYQADNE is excluded by problemsToRun"
methods, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_box_methods()
print(f"{len(problems)} box-constrained problems, {len(methods)} methods")

In [ ]:
run(
    problems,
    methods,
    precision,
    noise,
    ERROR_CAUSING_TASKS=[],
    TL=time_limit,
    boxed=True,
)

In [ ]:
from qnlab.experiment.for_cutest_vis import individual_plot

alg_names, callsM, fxsM, gnormsM, problems = load_results(
    methods, problems, precision, noise, gtol, boxed=True
)
draw_pp(
    alg_names,
    callsM,
    ALGORITHM_COLORS,
    ALGORITHM_LINE_STYLES,
    precision,
    noise,
    gtol,
    boxed=True,
)

if gtol == 1e-5:
    individual_plot(problems, methods, precision, noise, boxed=True)
if True:
    data = {"problem": problems}
    for i, alg_name in enumerate(alg_names):
        data[alg_name] = callsM[i, :].tolist()
    df = pd.DataFrame(data).set_index("problem")

    def color_scale_with_cmap(row):
        if np.all(np.isinf(row.values)):
            return ["background-color: rgba(0, 0, 0, 0.8)" for _ in row.values]
        norm = plt.Normalize(vmin=row.min(), vmax=row.min() * 10)
        cmap = matplotlib.colormaps["coolwarm"]
        return [
            f"background-color: rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, 0.8)"
            for r, g, b, _ in cmap(norm(row.values))
        ]

    display(df.style.apply(color_scale_with_cmap, axis=1))